# Lab - Tool Errors: Categories, Retries and Flags

Three tasks on what a tool returns when things go wrong - and who handles it. A tool that fails silently, or crashes the loop, leaves both Claude and the customer without an answer.

| Task | What you learn |
| --- | --- |
| 1 | Not found is data; every real failure comes out of one five-field helper |
| 2 | A business error is final, a transient error may pass: category and retryable |
| 3 | Retry the retryable in code; send the rest to Claude flagged is_error |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
SYSTEM = (
    "You are ShopAssist, an online store's support assistant. "
    "Use the tools to look up orders and to start returns. "
    "The start_return tool checks the return policy; do not "
    "judge it yourself. Answer in one or two sentences."
)

ORDERS = {
    "ORD-12345678": {"item": "jacket", "status": "delivered", "delivered_days_ago": 20},
    "ORD-87654321": {"item": "lamp", "status": "delivered", "delivered_days_ago": 45},
}

TOOLS = [
    {"name": "lookup_order",
     "description": "Look up an order by its ID. Returns the item, the status and how "
                    "many days ago it was delivered.",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}},
                      "required": ["order_id"]}},
    {"name": "start_return",
     "description": "Start a return for a delivered order that is inside the 30-day "
                    "window. Returns the return label number.",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}},
                      "required": ["order_id"]}},
]

# A stand-in for the warehouse service that prints return labels. Like any network
# service, it sometimes times out.
WAREHOUSE = {"fail_next": 0}


def warehouse_create_label(order_id):
    if WAREHOUSE["fail_next"]:
        WAREHOUSE["fail_next"] -= 1
        raise TimeoutError("warehouse API timed out after 5 s")
    return "RET-2048"

---

## Task 1 - Not found is data; errors have one shape

In [ ]:
# ============================================================
# TASK 1 - Not found is data; errors have one shape
# ============================================================
#
# WHAT TO DO
#   Write tool_error(), the one helper every tool uses to report
#   a failure, and finish lookup_order(): an order that does not
#   exist is a normal answer, found False.
#
# WHY IT MATTERS
#   An unknown order number is not an error - it is an answer,
#   just an empty one. A real failure is different, and it is
#   reported as a structured error: a flag, a category, a
#   retryable flag, a message safe for the customer and one for
#   the logs. Built by one helper, every tool reports failures
#   the same way - a contract Claude and your logs can rely on.
#
# WHERE TO SEE IT IN THE LECTURE
#   "When the order does not exist, that is not an error".
#
# HOW TO DO IT
#   Three blanks:
#
#       True        every error says so in isError
#       retryable   whether trying again can help
#       False       the order was not found
# ============================================================

# TODO: replace each ... below with the value named beside it
def tool_error(category, retryable, customer, developer):
    return {"isError": ..., "errorCategory": category, "isRetryable": ...,   # True, retryable
            "customerMessage": customer, "developerMessage": developer}


def lookup_order(order_id):
    order = ORDERS.get(order_id)
    if order is None:
        return {"found": ..., "message": "No order with this ID."}   # False
    return {"found": True, "order": order}

print(lookup_order("ORD-99999999"))
print(tool_error("business", False, "Sorry, I can't do that.", "rule X failed"))

check("not_found_data", lookup_order=lookup_order, tool_error=tool_error)

---

## Task 2 - Business errors and transient errors

In [ ]:
# ============================================================
# TASK 2 - Business errors and transient errors
# ============================================================
#
# WHAT TO DO
#   Finish start_return(). An order past the window is refused
#   with a business error, not retryable. When the warehouse
#   service times out, report a transient error that is
#   retryable. Otherwise return the label.
#
# WHY IT MATTERS
#   Both are failures, and they mean opposite things. The policy
#   refusal will be the same answer every time: retrying is
#   pointless, the customer needs an explanation. The timeout
#   may pass in a second: retrying is exactly right. The
#   category and the retryable flag carry that difference to
#   whoever handles the result.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The start return tool is different".
#
# HOW TO DO IT
#   Three blanks:
#
#       "business"    the policy refused the action
#       "transient"   the service failed, for now
#       True          a transient error may succeed on retry
# ============================================================

# TODO: replace each ... below with the value named beside it
def start_return(order_id):
    order = ORDERS[order_id]
    if order["delivered_days_ago"] > 30:
        return tool_error(..., False,   # "business"
                          "This order is past the 30-day return window.",
                          f"delivered_days_ago={order['delivered_days_ago']} > 30")
    try:
        label = warehouse_create_label(order_id)
    except TimeoutError as exc:
        return tool_error(..., ...,   # "transient", True
                          "Our warehouse is slow right now. Please try again in a moment.",
                          f"warehouse_create_label: {exc}")
    return {"return_label": label, "order_id": order_id}

print(start_return("ORD-87654321"))
WAREHOUSE["fail_next"] = 1   # provided: the next warehouse call times out
print(start_return("ORD-12345678"))
WAREHOUSE["fail_next"] = 0

check("error_categories", start_return=start_return, warehouse=WAREHOUSE)

---

## Task 3 - Retry in code, flag what is left

In [ ]:
# ============================================================
# TASK 3 - Retry in code, flag what is left
# ============================================================
#
# WHAT TO DO
#   Finish run_tool(): it runs a tool and, when the result is a
#   retryable error, runs it once more. The loop sets is_error on
#   every tool_result from the output's flag. Run two cases: the
#   jacket while the warehouse times out once, and the lamp.
#
# WHY IT MATTERS
#   A transient failure is your code's to absorb: retry it, and
#   Claude never needs to know. A business error is not - it
#   goes back to Claude flagged is_error, the flag the protocol
#   understands, and Claude explains it in the words of the
#   customer message. The developer message never reaches the
#   customer.
#
# WHERE TO SEE IT IN THE LECTURE
#   "In the tool result, we also set the is error flag".
#
# HOW TO DO IT
#   Three blanks:
#
#       2               the most runs of one tool call
#       "isRetryable"   retry only what may succeed next time
#       "isError"       the flag the tool_result carries
# ============================================================

TOOL_FUNCTIONS = {"lookup_order": lookup_order, "start_return": start_return}
ATTEMPTS = {}

# TODO: replace each ... below with the value named beside it
def run_tool(name, args):
    for attempt in range(...):   # 2 - the most runs
        ATTEMPTS[name] = ATTEMPTS.get(name, 0) + 1
        output = TOOL_FUNCTIONS[name](**args)
        if not (output.get("isError") and output.get(...)):   # "isRetryable"
            break
    return output


def handle(text):
    messages = [{"role": "user", "content": text}]
    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=500, output_config={"effort": "low"},
            system=SYSTEM, tools=TOOLS, messages=messages)
        if response.stop_reason != "tool_use":
            return response, messages
        messages.append({"role": "assistant", "content": response.content})
        results = []
        for block in response.content:
            if block.type == "tool_use":
                output = run_tool(block.name, block.input)
                print(f"  {block.name} -> {output}")
                results.append({"type": "tool_result", "tool_use_id": block.id,
                                "content": json.dumps(output),
                                "is_error": bool(output.get(...))})   # "isError"
        messages.append({"role": "user", "content": results})

cases = []
try:
    for text, timeouts in (("I want to return the jacket from order ORD-12345678.", 1),
                           ("I want to return the lamp from order ORD-87654321.", 0)):
        ATTEMPTS.clear()
        WAREHOUSE["fail_next"] = timeouts   # provided: the warehouse times out this often
        print("Customer:", text)
        response, messages = handle(text)
        print("ShopAssist:", response.content[0].text, "| runs:", dict(ATTEMPTS), "\n")
        cases.append((response, messages, dict(ATTEMPTS)))
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("retry_transient", cases=cases)

---

## Done

Not found is data. A real failure is a structured error with a category and a retryable flag: recover in code what code can recover, and hand the rest to Claude flagged is_error, with a customer message it can use and a developer message for your logs.